---
title: "Human Review and Interrupts"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval, reliability]
---

Review is a workflow transition, not a disclaimer added after generation. The Change Planner pauses with its summary, unknowns, and allowed actions; a typed decision then approves, edits, rejects, or sends the investigation back for more evidence.


## Pause and resume one investigation

The review node has no external side effect. `interrupt` persists the current thread, and `Command(resume=...)` supplies the validated decision when the same investigation resumes.


In [1]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from change_planner.fixtures import request_for
from change_planner.workflow import build_change_planner_graph, make_context

saver = InMemorySaver()
context = make_context()
config = {"configurable": {"thread_id": "dry-run-review"}}
graph = build_change_planner_graph(checkpointer=saver)
paused = graph.invoke(
    {"request": request_for("dry-run-01").model_dump(), "events": [], "branch_results": []},
    config=config,
    context=context,
    version="v2",
)
review_request = paused.interrupts[0].value
print({"summary": review_request["summary"], "actions": review_request["allowed_actions"]})
completed = graph.invoke(
    Command(resume={"action": "approve", "reason": "evidence and unknowns reviewed"}),
    config=config,
    context=context,
    version="v2",
)
print(completed.value["status"], context.controller.effects)
assert completed.value["status"] == "complete"
assert context.controller.effects[-1] == "export:plan"


{'summary': 'Add a dry-run branch before notebook mutation while preserving the existing output and solution-cell contracts.', 'actions': ['approve', 'edit', 'reject', 'request_evidence']}
complete ['search:operations', 'search:behavior', 'search:tests', 'search:history', 'test:test-link:fixture/change-cli@8f2c1d:tests/test_clear_outputs.py', 'export:plan']


The thread resumes after the review decision and exports exactly once. The checkpoint contains workflow state; the effect ledger independently records that the export did not happen before approval and was not duplicated after resume.

## Decisions change the artifact

An edit increments the plan version, a rejection enters a bounded revision, and a request for evidence revisits planning. These are control-flow differences, not comments attached to an otherwise identical answer.


In [2]:
from change_planner.schemas import ReviewDecision
from change_planner.workflow import run_fixture

edited = run_fixture(
    "dry-run-01",
    review_decision=ReviewDecision(
        action="edit",
        reason="state the scope explicitly",
        edited_summary="Review the dry-run boundary before implementation.",
    ),
)
print({"status": edited["status"], "version": edited["artifact"]["version"], "summary": edited["artifact"]["summary"]})
assert edited["status"] == "complete"
assert edited["artifact"]["version"] == 2
assert edited["run_metrics"]["effects"].count("export:plan") == 1


{'status': 'complete', 'version': 2, 'summary': 'Review the dry-run boundary before implementation.'}


The review decision is part of the artifact lineage. Chapter 07 moves the same protocol across a process boundary and separates checkpoint state from durable repository memory.
